# Learn the physics, then solve it exactly: IC-ThermBench (report Sec 9.31)

A small U-Net reads only the power-free inputs (layout, conductivity, cooling) and outputs a local 2D conductance
field: lateral conductivity per cell and conductance to ambient per cell. Temperature is the exact solution of the
discrete conservation law with those conductances (preconditioned CG, implicit gradients;
`src/hybrid/learned_conductance.py`). Every prediction is a genuine steady-state solution and exactly linear in power.

**Hypothesis.** Inferring *local physics* transfers to unseen packages (S5) better than predicting temperature
directly, and needs far fewer samples. Bars: ThermoNO S5 6.70 +- 0.37 K (3 seeds), Therm-FM 15.51 K, U-Net 19.10 K.

**Protocol.** IC-ThermBench's split, their vendored metrics, selection on their val split, S5 zero-shot from the
frozen S4 model. `--train-n` caps the training set (val/test stay full). Attach `rajulkabir/ic-thermbench-s2-s5`.

In [ ]:
import subprocess, sys, torch
print('PyTorch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'h5py', 'pyyaml'], check=True)

In [ ]:
import shutil, zipfile
from pathlib import Path
REPO = 'https://github.com/rajul-kk/thermo-3dic-surrogates.git'
CLONE, WORK = Path('/kaggle/working/repo'), Path('/kaggle/working')
OUT = WORK / 'phys_results'; OUT.mkdir(parents=True, exist_ok=True)
shutil.rmtree(CLONE, ignore_errors=True)
r = subprocess.run(['git', 'clone', '--depth', '1', REPO, str(CLONE)], capture_output=True, text=True)
assert r.returncode == 0, r.stderr
print('repo at', subprocess.run(['git', '-C', str(CLONE), 'rev-parse', '--short', 'HEAD'],
                                capture_output=True, text=True).stdout.strip())
mats = list(Path('/kaggle/input').rglob('level2_steady/input.mat'))
if mats:
    DATA = mats[0].parent.parent
else:
    z = next(Path('/kaggle/input').rglob('s2_s5_datasets.zip'))
    zipfile.ZipFile(z).extractall(WORK / 'icb')
    DATA = next((WORK / 'icb').rglob('level2_steady/input.mat')).parent.parent
print('data root', DATA)

In [ ]:
# (tag, model, scope, extra args). Most important first: a session timeout still leaves the key result.
S5 = ['--transfer', 'level5']
RUNS = [('phys_full', 'physsolve', 'level4', S5),
        ('phys_n1080', 'physsolve', 'level4', S5 + ['--train-n', '1080']),
        ('thermono_n1080', 'thermono', 'level4', S5 + ['--train-n', '1080']),
        ('phys_n108', 'physsolve', 'level4', S5 + ['--train-n', '108', '--epochs', '200', '--min-epochs', '150']),
        ('thermono_n108', 'thermono', 'level4', S5 + ['--train-n', '108', '--epochs', '200', '--min-epochs', '150']),
        ('phys_s2', 'physsolve', 'level2', [])]
EXTRA = ['--epochs', '60', '--min-epochs', '45', '--patience', '15', '--lr', '2e-3']
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
import time
for tag, model, scope, more in RUNS:
    out = OUT / f'{tag}.json'
    if out.exists():
        print(tag, 'already done'); continue
    t = time.time()
    args = EXTRA + more            # later flags win in argparse, so per-run epochs override EXTRA
    cmd = [sys.executable, 'scripts/icb_physsolve.py', '--model', model, '--scope', scope, '--data-root', str(DATA),
           '--device', DEVICE, '--out', str(out)] + args
    print('===', tag, ' '.join(more), flush=True)
    p = subprocess.Popen(cmd, cwd=CLONE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if 'Warning' not in line:
            print('  ' + line, end='', flush=True)
    if p.wait() != 0:
        print(f'  !! {tag} failed, continuing'); continue
    print(f'  ({(time.time() - t) / 60:.1f} min)')

In [ ]:
import json
print(f"{'run':<16}{'n_train':>8}{'S4/S2 RMSE':>12}{'S5 RMSE':>10}{'S5 peak':>9}{'min':>6}")
for tag, model, scope, more in RUNS:
    f = OUT / f'{tag}.json'
    if not f.exists(): continue
    r = json.loads(f.read_text())
    s5 = r.get('level5', {})
    print(f"{tag:<16}{r['n_train']:>8}{r[scope]['rmse']:>12.3f}{s5.get('rmse', float('nan')):>10.3f}"
          f"{s5.get('max_temperature_error', float('nan')):>9.2f}{r['minutes']:>6.0f}")
print('bars: S5 Therm-FM 15.51, U-Net 19.10, ThermoNO(full) 6.70; S4 Therm-FM 0.933; S2 Therm-FM 0.443')